<a href="https://colab.research.google.com/github/M-Khalid16/PhotonicsAILab_projects/blob/main/Important_codes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np, math
from scipy import special, integrate

def TE_Rays():
    n0, n1, n2 = 1.0, 1.5, 1.48
    r = 50e-6
    lam = 1550e-9
    c = 299792458.0

    # Acceptance
    NA = math.sqrt(n1*n1 - n2*n2)
    alpha_max = math.asin(NA)
    rng = np.random.default_rng(2025)

    # Lambertian launch
    N = 50_000
    steps = 1000
    sigma = math.radians(0.2)
    u = rng.random(N, dtype=np.float64)
    alpha = np.arccos(1.0 - u * (1.0 - math.cos(alpha_max)))
    # Clip inside [-1,1] for numerical safety
    s = np.clip((n0 / n1) * np.sin(alpha), -1.0, 1.0)
    theta = np.arcsin(s)
    phi_c = math.asin(n2 / n1)
    theta_max = math.pi / 2 - phi_c

    alive = np.ones(N, dtype=bool)
    th = theta.copy()
    for _ in range(steps):
        th += rng.normal(0.0, sigma, size=N)
        th = np.maximum(th, 0.0)
        alive &= (th < theta_max)

    N_surv = int(alive.sum())

    # TE0 mode power fraction via Simpson rule
    x = np.linspace(0.0, r, 10_000, dtype=np.float64)
    u0 = 2.404825557695773  # first root of J0
    E = special.jv(0, u0 * x / r)
    integrand = (E**2) * (2.0 * math.pi * x)
    P_te = integrate.simpson(integrand, x=x)   # <- returns a single float
    P_air = math.pi * r * r
    eta_te = P_te / P_air

    return round(N_surv * eta_te, 6)

print(TE_Rays())



4753.959632


A 100 km step-index silica fiber has core index $n_1 = 1.5000$, cladding index $n_2 = 1.4800$, core radius $r = 50\,\mu\text{m}$, and operates at wavelength $\lambda = 1550\,\text{nm}$. Light is launched from air $n_0 = 1.0$ through a flat-polished facet. Rays are launched with polar angle $\alpha$ drawn from the Lambertian facet law,  with probability density $p(\alpha) \propto \sin\alpha$ for $\alpha \in [0,\alpha_{\max}]$. You need to compute the internal meridional angle $\theta \in [0,\theta_{\max})$ in radians, using double precision `float64`. The critical angle at the core-cladding interface is $ \phi_c$.

We know that a ray of light is guided in the fiber if its incidence angle satisfies $\phi > \phi_c$. For meridional rays, $\phi = \tfrac{\pi}{2} - \theta$, therefore, loss occurs when $\theta \ge \theta_{\max},
\qquad
\theta_{\max} = \tfrac{\pi}{2} - \phi_c.
$
Divide the 100 km fiber into $k$ = 1000 independent segments. If at any segment boundary $\theta_k \ge \theta_{\max}$, the ray is lost for the remainder of the simulation. Do not clip upper values except for this loss check, only enforce the lower bound $\max(0,\cdot)$. You are supposed to simulate exactly $N = 50{,}000$ independent rays using NumPy's `Generator(PCG64)` seeded with 2025. For this purpose, use fully vectorized operations and no Python loops over rays.  All values must be `float64`.

In addition, every surviving ray at the output face excites a normalized TE modal field distribution:
$
E_\mathrm{TE}(r_\perp) = J_0\!\left( \frac{u\,r_\perp}{r} \right)$ where $J_0$ is the Bessel function of the first kind and $u$ is the first root of $J_0$. The total guided optical power is proportional to
$
P_\text{guided} = \int_0^{r} |E_\mathrm{TE}(r_\perp)|^2\, 2\pi r_\perp\,dr_\perp.
$
Compute both the number of surviving rays at the output $N_\mathrm{surv}$ and the normalized TE-mode power fraction. Define the TE-weighted survival index using NumPy RNG `Generator(PCG64)` with seed = 2025.  
Perform a numeric integration using Simpson's rule over at least 10,000 points. Return one floating-point number for TE-weighted survival index i.e. $I_\mathrm{TE}$ rounded to four decimal places. Any mismatch beyond 0.0001 is onsidered incorrect.



In [ ]:
import numpy as np

# =========================
# HARD-CODED CONFIGURATION
# =========================
SEED = 12345
np.random.seed(SEED)

# Wavelength domain (nm)
LAMBDA_MIN = 400.0
LAMBDA_MAX = 700.0
N = 131072  # large power-of-two grid (computationally heavy)
wavelengths = np.linspace(LAMBDA_MIN, LAMBDA_MAX, N)
dl = wavelengths[1] - wavelengths[0]

# Target wavelength (nm)
TARGET_WAVELENGTH = 658.0

# ---- Ground-truth synthetic spectrum (hard-coded, deterministic) ----
# Sparse "line" spectrum with many narrow components
line_centers_nm = np.array([
    412.5, 430.1, 447.7, 470.0, 486.1, 501.6, 517.0, 532.8, 546.1, 577.0,
    589.0, 589.6, 602.0, 612.9, 623.4, 640.2, 656.28, 660.0, 667.8, 682.0,
    693.1
], dtype=float)
line_amps = np.array([
    0.35, 0.48, 0.40, 0.55, 0.80, 0.45, 0.62, 0.50, 0.58, 0.42,
    0.95, 0.82, 0.33, 0.28, 0.40, 0.52, 1.00, 0.44, 0.39, 0.31,
    0.27
], dtype=float) * 5.0  # scale up
line_sigma_nm = 0.08  # very narrow intrinsic width

# Smooth broadband background (sum of wide Gaussians)
bb_centers = np.array([440.0, 520.0, 610.0])
bb_amps = np.array([0.6, 0.8, 0.5]) * 1.2
bb_sigmas = np.array([40.0, 55.0, 45.0])

# Instrument response
# True PSF: mixture of two Gaussians (asymmetry)
psf_sigma_nm_true = 1.20
psf_sigma_nm_true2 = 3.50
psf_mix_true = 0.78  # mixture weight

# Noise level (relative)
NOISE_STD = 0.012

# =========================
# HARD-CODED MEASUREMENT
# =========================
def gaussian(x, mu, sigma):
    z = (x - mu) / sigma
    return np.exp(-0.5 * z * z)

# Ground-truth high-res spectrum s_true (nonnegative)
s_true = np.zeros_like(wavelengths)
# narrow lines
for mu, A in zip(line_centers_nm, line_amps):
    s_true += A * gaussian(wavelengths, mu, line_sigma_nm)
# broadband background
for mu, A, sig in zip(bb_centers, bb_amps, bb_sigmas):
    s_true += A * gaussian(wavelengths, mu, sig)
# mild baseline tilt
s_true += 0.02 * (wavelengths - LAMBDA_MIN) / (LAMBDA_MAX - LAMBDA_MIN)

# True instrument PSF kernel on same grid (centered at zero)
half_span_nm = 6.0
M = int(np.ceil(half_span_nm / dl))
grid_local = np.arange(-M, M + 1) * dl  # nm offsets
h_true_local = (
    psf_mix_true * gaussian(grid_local, 0.0, psf_sigma_nm_true) +
    (1.0 - psf_mix_true) * gaussian(grid_local, 0.0, psf_sigma_nm_true2)
)
h_true_local /= (h_true_local.sum() * dl)  # unit area

# Embed local PSF into full-length array for FFT convolution
h_true = np.zeros_like(wavelengths)
center = N // 2
start = center - M
h_true[start:start + h_true_local.size] = h_true_local
# circularly shift so that FFT sees zero-centered kernel
h_true = np.roll(h_true, -center)

# Convolution via FFT (measurement y = h * s + noise)
S = np.fft.rfft(s_true)
H = np.fft.rfft(h_true)
y_clean = np.fft.irfft(S * H, n=N)
noise = NOISE_STD * np.max(y_clean) * np.random.randn(N)
y = np.clip(y_clean + noise, 0.0, None)

# =========================
# INVERSE PROBLEM (BLIND)
# =========================
# We estimate s (spectrum) and h (PSF) jointly:
#   min_{s>=0, h>=0, sum(h)dl=1}  0.5||y - h*s||_2^2 + λ1||s||_1 + λTV * TV(s)
# with alternating updates:
#   - Fix h, update s via proximal gradient (FFT conv, L1 + TV proximals)
#   - Fix s, update h via projected gradient (nonnegativity + unit-area)

# Regularization strengths
L1 = 0.015
LTV = 0.12

# Steps and iterations
ITERS_ALT = 200         # outer alternations (heavy)
ITERS_S = 4             # inner s-updates per alternation
ITERS_H = 1             # one h-step per alternation
STEP_S = 0.9            # s gradient step (relative to Lipschitz guess)
STEP_H = 0.5            # h gradient step

init_sigma = 2.5
h_local = gaussian(grid_local, 0.0, init_sigma)
h_local /= (h_local.sum() * dl)
h = np.zeros_like(wavelengths)
h[start:start + h_local.size] = h_local
h = np.roll(h, -center)

# Initialize s as deconvolved Wiener-like guess (simple)
eps = 1e-6
H0 = np.fft.rfft(h)
Y = np.fft.rfft(y)
S_wiener = Y * np.conj(H0) / (np.abs(H0)**2 + 1e-4)
s = np.maximum(np.fft.irfft(S_wiener, n=N), 0.0)

# Helpers for TV prox (1D ROF via Chambolle)
def tv_denoise_chambolle_1d(f, weight, n_iter=25):
    # Classic Chambolle’s algorithm adapted for 1D TV
    p = np.zeros_like(f)
    g = np.zeros_like(f)
    tau = 0.125
    for _ in range(n_iter):
        # divergence of p (with Neumann BC)
        div_p = np.empty_like(p)
        div_p[0] = p[0]
        div_p[1:-1] = p[1:-1] - p[0:-2]
        div_p[-1] = -p[-2]
        g = f - weight * div_p
        # gradient of g
        grad_g = np.empty_like(g)
        grad_g[0:-1] = g[1:] - g[0:-1]
        grad_g[-1] = 0.0
        # update p
        p += (tau / weight) * grad_g
        p = p / np.maximum(1.0, np.abs(p))
    # final reconstruction
    div_p = np.empty_like(p)
    div_p[0] = p[0]
    div_p[1:-1] = p[1:-1] - p[0:-2]
    div_p[-1] = -p[-2]
    u = f - weight * div_p
    return u

# Soft-thresholding
def soft_thresh(x, lam):
    return np.sign(x) * np.maximum(np.abs(x) - lam, 0.0)

# Precompute frequency grid for FFTs
def conv_fft(x, h_fft=None, h_spatial=None):
    if h_fft is None:
        h_fft = np.fft.rfft(h_spatial)
    X = np.fft.rfft(x)
    return np.fft.irfft(X * h_fft, n=N)

for k in range(ITERS_ALT):

    Hk = np.fft.rfft(h)
    for _ in range(ITERS_S):
        # Gradient of 0.5||y - h*s||^2 = (h^T * (h*s - y))
        hs = np.fft.irfft(np.fft.rfft(s) * Hk, n=N)
        resid = hs - y
        grad = np.fft.irfft(np.fft.rfft(resid) * np.conj(Hk), n=N)
        s = s - STEP_S * grad
        # Prox L1
        s = soft_thresh(s, STEP_S * L1)
        # Prox TV (Chambolle denoise as proximal)
        s = tv_denoise_chambolle_1d(s, STEP_S * LTV, n_iter=10)
        # Nonnegativity
        s = np.maximum(s, 0.0)


    # Gradient wrt h: ( (h*s - y) * s^T ) in convolutional sense
    Sk = np.fft.rfft(s)
    hs = np.fft.irfft(Sk * Hk, n=N)
    resid = hs - y
    # grad_h = conv(resid, flip(s))
    grad_h = np.fft.irfft(np.fft.rfft(resid) * np.conj(Sk), n=N)
    h = h - STEP_H * grad_h
    # Enforce support to local window only (compact PSF)
    h_new = np.zeros_like(h)
    h_local_est = np.roll(h, center)[start:start + h_true_local.size]
    h_local_est = np.maximum(h_local_est, 0.0)
    # normalize area to 1
    area = np.sum(h_local_est) * dl
    if area <= 1e-12:
        h_local_est[:] = 0.0
        h_local_est[M] = 1.0 / dl  # reset to delta if degenerate
    else:
        h_local_est /= area
    h_new[start:start + h_local_est.size] = h_local_est
    h = np.roll(h_new, -center)

# =========================
# OUTPUT: single float
# =========================
# Reconstructed intensity at 658 nm from s (deconvolved estimate)
idx = int(np.argmin(np.abs(wavelengths - TARGET_WAVELENGTH)))
val = float(s[idx])
# Print ONLY the floating-point number:
print(f"{val:.10f}")


98183.2915482840


Consider a scenario in optical communication where a transmitted optical signal, represented as a spectrum across at least one hundred thousand wavelength points $N$ between 400 nm and 700 nm, passes through an optical channel with an unknown response function $h$. The received signal represented as $y$ is a noisy, blurred version of the transmitted spectrum due to the convolution with this channel response.  Generate a synthetic “true” optical spectrum $s_{true}$ consisting of narrow spectral lines plus smooth broadband components. Your task is to jointly reconstruct both the original spectrum and the channel response $h_{true}$ by formulating and solving a blind inverse problem. The solution should enforce realistic constraints like non-negativity of the transmitted spectrum, sparsity of spectral lines, smoothness of background components, and unit-area normalization of the channel response. The process involves iterative optimization with many updates, using sparsity-promoting penalties and total-variation regularization approaches, and relies on convolution operations carried out over the full dataset to ensure the task is computationally demanding.  The optimization uses $L_1$ regularization (parameter L1) for sparsity and total variation regularization (parameter LTV) for smoothness. In the end, report only a single floating quantity which is the reconstructed signal intensity $s[idx]$ at a wavelength of 658 nm, considering random seed as 12345 for reproducibility. Your answer should be accurate up to 10 decimal points.

An optical engineer is interested in designing a Hollow core silica based Photonic Crystal Fiber (HC-PCF) for transmitting data efficiently over a long-haul optical communications system. The PCF is an air-filled circular core of diameter $6~\mu\text{m}$ (radius $a=3~\mu\text{m}$) and an overall cladding diameter of $125~\mu\text{m}$. The cladding consists of five photonic bandgap (PBG) layers with lattice pitch $\Lambda = 2.3~\mu\text{m}$.
The system operates at a wavelength $\lambda = 1.55~\mu\text{m}$ within a bandgap transmission window so that modes below cutoff are effectively guided in the designed PCF. The specification of five PBG layers and $\Lambda = 2.3~\mu\text{m}$ is provided to characterize the PCF geometry; for this calculation, treat guidance using the capillary cutoff model (air core, $n \!\approx\! 1$), neglect dispersion/leakage outside the bandgap window, and ignore degeneracies. Now if the engineer models the the PCF core as a circular capillary, what would be the number of transverse electric (TE) (i.e., $J_m'\!\left(x_{mn}^{\mathrm{TE}}\right) = 0$) and transverse magnetic (TM) (i.e., $J_m\!\left(x_{mn}^{\mathrm{TM}}\right) = 0$) modes that satisfy the cutoff condition for propagating through the fiber.  Also report the total number of propagating modes for a $1000\,\text{km}$ length of this fiber in such a way that the TE, TM and total mode count appears in one single line?



In [ ]:
import math

# --------------------------- Bessel J_m via series ----------------------------
def bessel_j_int(m, x, tol=1e-12, max_terms=2000):
    """Bessel J_m(x) for integer m using power series expansion."""
    if m < 0:
        mpos = -m
        jmpos = bessel_j_int(mpos, x, tol, max_terms)
        return jmpos if (mpos % 2 == 0) else -jmpos

    if x == 0.0:
        return 1.0 if m == 0 else 0.0

    halfx = 0.5 * x
    term = (halfx ** m) / float(math.factorial(m))
    s = term
    k = 1
    while k < max_terms:
        ratio = - (halfx * halfx) / float(k * (k + m))
        term *= ratio
        s += term
        if abs(term) < tol * (1.0 + abs(s)):
            break
        k += 1
    return s

def bessel_j_deriv_int(m, x, tol=1e-12):
    """Derivative J_m'(x) using recurrence."""
    return 0.5 * (bessel_j_int(m - 1, x, tol) - bessel_j_int(m + 1, x, tol))

# --------------------------- Root finding utilities ---------------------------
def sign(v):
    if v > 0.0: return 1
    if v < 0.0: return -1
    return 0

def find_brackets(f, x_min, x_max, samples):
    brackets = []
    step = (x_max - x_min) / float(samples)
    prev_x = x_min
    prev_y = f(prev_x)
    for i in range(1, samples + 1):
        x = x_min + i * step
        y = f(x)
        sp = sign(prev_y)
        sc = sign(y)
        if sp == 0: sp = 1
        if sc == 0: sc = -1
        if sp != sc:
            brackets.append((prev_x, x))
        prev_x, prev_y = x, y
    return brackets

def bisect_root(f, a, b, tol=1e-10, itmax=80):
    fa, fb = f(a), f(b)
    if fa == 0.0: return a
    if fb == 0.0: return b
    if fa * fb > 0: return None
    for _ in range(itmax):
        c = 0.5 * (a + b)
        fc = f(c)
        if abs(fc) < tol or 0.5 * (b - a) < tol:
            return c
        if fa * fc < 0:
            b, fb = c, fc
        else:
            a, fa = c, fc
    return 0.5 * (a + b)

def roots_in_interval(f, x_max, samples=2000, tol=1e-10):
    if x_max <= 0:
        return []
    brackets = find_brackets(f, 1e-12, x_max, samples)
    roots = []
    for (a, b) in brackets:
        r = bisect_root(f, a, b, tol=tol)
        if r is not None and 0 < r < x_max:
            if not any(abs(rr - r) < 1e-6 for rr in roots):
                roots.append(r)
    roots.sort()
    return roots

# ------------------------ Mode counting (capillary model) ---------------------
def count_capillary_modes(a_um, wavelength_um, samples_per_m=2000):
    """Return dict with TE/TM mode counts and lists of roots."""
    k0 = 2.0 * math.pi / float(wavelength_um)
    x_max = k0 * float(a_um)
    TE_list, TM_list = [], []
    m_max = int(x_max) + 10
    for m in range(0, m_max + 1):
        def f_tm(x, m_local=m): return bessel_j_int(m_local, x)
        def f_te(x, m_local=m): return bessel_j_deriv_int(m_local, x)
        tm_roots = roots_in_interval(f_tm, x_max, samples=samples_per_m)
        te_roots = roots_in_interval(f_te, x_max, samples=samples_per_m)
        for n, r in enumerate(tm_roots, start=1):
            TM_list.append((m, n, r))
        for n, r in enumerate(te_roots, start=1):
            TE_list.append((m, n, r))
    return {
        'n_total': len(TE_list) + len(TM_list),
        'n_TE': len(TE_list),
        'n_TM': len(TM_list),
        'k0a': x_max
    }

# ------------------------------ Main function ---------------------------------
def main():
    core_radius_um = 3.0
    wavelength_um = 1.55
    res = count_capillary_modes(core_radius_um, wavelength_um)

    # Updated to match the markdown task output format
    print(f"TE={res['n_TE']}, TM={res['n_TM']}, Total={res['n_total']}")

if __name__ == "__main__":
    main()


TE=23, TM=17, Total=40
